## Embedding Throughput Benchmark
Measure SentenceTransformer encoding throughput on CPU threads versus Metal (if available).

In [4]:
import os
import time
from statistics import mean
from typing import Dict, List, Optional

import torch
import pandas as pd
from sentence_transformers import SentenceTransformer

model_name = "all-MiniLM-L6-v2"
sentence_counts = [64, 256, 512, 1024]
batch_size = 64
cpu_threads = 8
repetitions = 3

DEFAULT_THREADS = torch.get_num_threads()
DEFAULT_OMP = os.environ.get("OMP_NUM_THREADS")
DEFAULT_MKL = os.environ.get("MKL_NUM_THREADS")
DEFAULT_TEMPLATE = "The weather is lovely today."

_model_cache: Dict[str, SentenceTransformer] = {}

def get_sentence_transformer(device: str) -> SentenceTransformer:
    if device not in _model_cache:
        _model_cache[device] = SentenceTransformer(model_name, device=device)
    return _model_cache[device]

def base_sentence() -> str:
    if "sentences" in globals() and sentences:
        return sentences[0]
    return DEFAULT_TEMPLATE

def make_sentences(count: int, template: Optional[str] = None) -> List[str]:
    template = template or base_sentence()
    return [f"{template} #{i}" for i in range(count)]

def configure_cpu_threads(num_threads: Optional[int]) -> None:
    if num_threads is None:
        torch.set_num_threads(DEFAULT_THREADS)
        if DEFAULT_OMP is not None:
            os.environ["OMP_NUM_THREADS"] = DEFAULT_OMP
        else:
            os.environ.pop("OMP_NUM_THREADS", None)
        if DEFAULT_MKL is not None:
            os.environ["MKL_NUM_THREADS"] = DEFAULT_MKL
        else:
            os.environ.pop("MKL_NUM_THREADS", None)
    else:
        torch.set_num_threads(num_threads)
        os.environ["OMP_NUM_THREADS"] = str(num_threads)
        os.environ["MKL_NUM_THREADS"] = str(num_threads)

def benchmark_device(device: str, label: str, num_threads: Optional[int] = None) -> List[Dict[str, float]]:
    if device == "cpu":
        configure_cpu_threads(num_threads)
    model = get_sentence_transformer(device)
    results: List[Dict[str, float]] = []
    for count in sentence_counts:
        samples = make_sentences(count)
        measurements = []
        for _ in range(repetitions):
            start = time.perf_counter()
            model.encode(samples, batch_size=batch_size, show_progress_bar=False, convert_to_numpy=True)
            measurements.append(time.perf_counter() - start)
        elapsed = mean(measurements)
        results.append({
            "config": label,
            "device": device,
            "sentences": count,
            "batch_size": batch_size,
            "elapsed_sec": elapsed,
            "throughput_sps": count / elapsed if elapsed else float("inf"),
        })
    return results

experiments = [
    {"device": "cpu", "label": f"CPU-{cpu_threads}-threads", "num_threads": cpu_threads},
    {"device": "cpu", "label": "CPU-1-thread", "num_threads": 1},
]

if torch.backends.mps.is_available():
    experiments.append({"device": "mps", "label": "MPS", "num_threads": None})
else:
    print("Metal (MPS) backend not available; ensure PyTorch is built with MPS support.")

records: List[Dict[str, float]] = []
for exp in experiments:
    records.extend(benchmark_device(**exp))

configure_cpu_threads(None)

results_df = pd.DataFrame(records).sort_values(["config", "sentences"])
display(results_df)

summary_df = results_df.groupby("config", as_index=False)["throughput_sps"].max().rename(columns={"throughput_sps": "peak_throughput_sps"})
display(summary_df)

,config,device,sentences,batch_size,elapsed_sec,throughput_sps
4,CPU-1-thread,cpu,64,64,0.042692,1499.100633
5,CPU-1-thread,cpu,256,64,0.167353,1529.701073
6,CPU-1-thread,cpu,512,64,0.361097,1417.901179
7,CPU-1-thread,cpu,1024,64,0.793347,1290.734045
0,CPU-8-threads,cpu,64,64,0.127927,500.285753
1,CPU-8-threads,cpu,256,64,0.138879,1843.331798
2,CPU-8-threads,cpu,512,64,0.294859,1736.421537
3,CPU-8-threads,cpu,1024,64,0.599792,1707.259425
8,MPS,mps,64,64,0.038026,1683.038674
9,MPS,mps,256,64,0.045370,5642.533055


,config,peak_throughput_sps
0,CPU-1-thread,1529.701073
1,CPU-8-threads,1843.331798
2,MPS,5642.533055


### Sentence Length Scaling Benchmark
Measure how encoding latency changes as token counts per sentence increase while keeping the batch size fixed.

In [5]:
length_token_counts = [8, 16, 32, 64, 96, 128, 160, 192, 224]
sentences_per_length = 256
length_batch_size = 64

def build_sentence_with_tokens(token_count: int) -> str:
    token = "word"
    return " ".join([token] * max(1, token_count))

def make_length_samples(token_count: int, total: int) -> List[str]:
    sentence = build_sentence_with_tokens(token_count)
    return [sentence for _ in range(total)]

def benchmark_sentence_lengths(device: str, label: str, token_counts: List[int], num_threads: Optional[int] = None) -> List[Dict[str, float]]:
    if device == "cpu":
        configure_cpu_threads(num_threads)
    model = get_sentence_transformer(device)
    tokenizer = model._first_module().tokenizer
    max_seq_len = model.get_max_seq_length()
    seen_targets = set()
    results: List[Dict[str, float]] = []
    for token_count in token_counts:
        capped = min(token_count, max_seq_len - 4)
        if capped < 2:
            continue
        if capped in seen_targets:
            continue
        seen_targets.add(capped)
        samples = make_length_samples(capped, sentences_per_length)
        encodings = tokenizer(samples[:8], add_special_tokens=True, padding=False, truncation=True, max_length=max_seq_len)
        token_lengths = [len(ids) for ids in encodings["input_ids"]]
        avg_tokens = mean(token_lengths)
        max_tokens = max(token_lengths)
        measurements = []
        for _ in range(repetitions):
            start = time.perf_counter()
            model.encode(samples, batch_size=length_batch_size, show_progress_bar=False, convert_to_numpy=True)
            measurements.append(time.perf_counter() - start)
        elapsed = mean(measurements)
        results.append({
            "config": label,
            "device": device,
            "target_tokens": capped,
            "avg_tokens": avg_tokens,
            "max_tokens": max_tokens,
            "sentences": sentences_per_length,
            "batch_size": length_batch_size,
            "elapsed_sec": elapsed,
            "throughput_sps": sentences_per_length / elapsed if elapsed else float("inf"),
            "tokens_per_sec": (sentences_per_length * avg_tokens) / elapsed if elapsed else float("inf"),
        })
    return results

length_records: List[Dict[str, float]] = []
for exp in experiments:
    length_records.extend(benchmark_sentence_lengths(exp["device"], exp["label"], length_token_counts, exp.get("num_threads")))

configure_cpu_threads(None)

length_df = pd.DataFrame(length_records).sort_values(["config", "target_tokens"])
display(length_df)

length_summary = length_df.groupby("config", as_index=False)["tokens_per_sec"].max().rename(columns={"tokens_per_sec": "peak_tokens_per_sec"})
display(length_summary)

,config,device,target_tokens,avg_tokens,max_tokens,sentences,batch_size,elapsed_sec,throughput_sps,tokens_per_sec
9,CPU-1-thread,cpu,8,10,10,256,64,0.178140,1437.068833,14370.688331
10,CPU-1-thread,cpu,16,18,18,256,64,0.336188,761.477910,13706.602389
11,CPU-1-thread,cpu,32,34,34,256,64,0.672958,380.410023,12933.940790
12,CPU-1-thread,cpu,64,66,66,256,64,1.415352,180.873735,11937.666508
13,CPU-1-thread,cpu,96,98,98,256,64,1.878454,136.282263,13355.661777
14,CPU-1-thread,cpu,128,130,130,256,64,2.582141,99.142524,12888.528077
15,CPU-1-thread,cpu,160,162,162,256,64,3.272111,78.236961,12674.387626
16,CPU-1-thread,cpu,192,194,194,256,64,3.934984,65.057440,12621.143408
17,CPU-1-thread,cpu,224,226,226,256,64,4.758872,53.794265,12157.503793
0,CPU-8-threads,cpu,8,10,10,256,64,0.132001,1939.380878,19393.808781


,config,peak_tokens_per_sec
0,CPU-1-thread,14370.688331
1,CPU-8-threads,27116.865413
2,MPS,65968.998194
